# ExplainPhish: HTML Phishing Detection UTAR
This NOTEBOOK provides an introduction to the process of detecting phishing HTML documents using machine learning and explainable AI.

**Objective**: To predict whether an HTML document/attachment is Phishing (`1`) or Benign (`0`) based on static features extracted from the **CIC-Trap4Phish2025** benchmark.

**Evaluation metric**: ROC-AUC (Area Under the Receiver Operating Characteristic Curve) and F1-Score

## Contents
- [1.Setup](#scrollTo=a5KcDAE8CnmB)
- [2.Loading the Data](#scrollTo=tp0cW0CD11Hi&line=1&uniqifier=1)
- [3.Visualizing and Understanding the Data](#scrollTo=3NuP1zcmCnmF&line=1&uniqifier=1)
- [4.Preprocessing and Feature Creation](#scrollTo=rsPYkguwCnmO&line=1&uniqifier=1)
- [5.Building the Machine Learning Model](#scrollTo=FoKdK60PCnmP&line=1&uniqifier=1)
- [6.Creating prediction results](#scrollTo=tn_kdvWYCnmQ&line=2&uniqifier=1)

## 1.Setup

### 1.1 Import Libraries
Let's load basic libraries.
Other required libraries will be loaded when we explain them.
- numpy: Library for efficient numerical computation
- pandas: Library useful for data analysis
- matplotlib: Graph drawing library
- seaborn: Graph drawing library as well

In [1]:
# Importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [3]:
# Specify the directory where this notebook is located after %cd.
%cd "/content/drive/MyDrive/WhereThisNotebookIsLocated"

[WinError 3] The system cannot find the path specified: '/content/drive/MyDrive/WhereThisNotebookIsLocated'
D:\codingProject\ExplainPhish


Run the cell below to check if the path is correctly set.

In [4]:
import os
from pathlib import Path

# Automatically get the current working directory
current_dir = Path(os.getcwd())

# Define candidate paths for HTML_Top13_Features.csv
candidates = [
    current_dir / "data" / "html" / "HTML_Top13_Features.csv",
    current_dir / "googleCollab" / "data" / "html" / "HTML_Top13_Features.csv",
    Path("data/html/HTML_Top13_Features.csv"),
    Path("googleCollab/data/html/HTML_Top13_Features.csv"),
    Path("../data/html/HTML_Top13_Features.csv"),
    Path("../googleCollab/data/html/HTML_Top13_Features.csv"),
    Path("/content/drive/MyDrive/ExplainPhish/data/html/HTML_Top13_Features.csv"),
    Path("/content/HTML_Top13_Features.csv")
]

data_file = None
for p in candidates:
    if p.exists():
        data_file = p
        break

# Check if path exists
if data_file and data_file.exists():
    print(f"HTML dataset file exists and path is correctly set:\n{data_file.resolve()}")
else:
    print("HTML dataset file is missing or path is not correctly set.")

HTML dataset file exists and path is correctly set:
D:\codingProject\ExplainPhish\googleCollab\data\html\HTML_Top13_Features.csv


## 2.Loading the Data

### 2.1 Data Overview
Run the cell to load the dataset as `pd.DataFrame`.

We load the static HTML feature dataset from `HTML_Top13_Features.csv` (derived from the CIC-Trap4Phish2025 dataset). We also clean any trailing export artifacts (`Unnamed` columns) and perform a stratified 60/40 train/test split to prepare the dataset for analysis and model training.

**IMPORTANT:**<br>
**When you make modifications to preprocessing or model training, always make sure to run all cells from this cell.**

In [5]:
from sklearn.model_selection import train_test_split

# Load raw HTML dataset
raw_df = pd.read_csv(data_file)

# Remove trailing artifact columns (Unnamed columns from CSV export)
unnamed_cols = [c for c in raw_df.columns if "unnamed" in c.lower()]
df = raw_df.drop(columns=unnamed_cols).copy()

# Ensure TARGET alias exists for consistency with the tutorial pipeline
if "TARGET" not in df.columns and "label" in df.columns:
    df["TARGET"] = df["label"]

# Stratified 60% Train / 40% Test Split (aligned with ExplainPhish methodology)
train, test = train_test_split(
    df, test_size=0.40, stratify=df["TARGET"], random_state=42
)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

print(f"Total dataset loaded: {len(df):,} rows x {len(df.columns)} columns (after dropping {len(unnamed_cols)} artifact columns)")

Total dataset loaded: 19,997 rows x 16 columns (after dropping 12 artifact columns)


Before conducting a full-scale analysis, we will first review a brief overview of the data.

In [6]:
# Check train data
print(f"train shape: {train.shape}")
train.head(3)

train shape: (11998, 16)


,file_name,url_punct_char_count,tag_count,whitespace_ratio,entropy,form_count,embedded_js_count,html_whitespace_ratio,script_entropy,min_link_length,external_link_count,total_script_characters,internal_link_count,url_digit_count,label,TARGET
0,sample_06438.html,2095,756,0.416768,5.465578,3,73,0.066372,5.539064,8,221,89458,21,1011,0,0
1,sample_05043.html,107,106,0.345622,5.382703,1,1,0.034719,5.368758,5,15,7128,1,126,1,1
2,sample_04636.html,1681,1182,0.359943,5.399269,5,4,0.091763,5.496770,9,172,13840,32,2065,0,0


In [7]:
# Check test data
print(f"test shape: {test.shape}")
test.head(3)

test shape: (7999, 16)


,file_name,url_punct_char_count,tag_count,whitespace_ratio,entropy,form_count,embedded_js_count,html_whitespace_ratio,script_entropy,min_link_length,external_link_count,total_script_characters,internal_link_count,url_digit_count,label,TARGET
0,sample_04241.html,0,5,0.000000,5.375038,0,1,0.064619,5.309058,0,0,1713,0,0,1,1
1,sample_07098.html,525,383,0.360669,5.282739,5,23,0.103680,5.381425,19,61,28705,7,23,0,0
2,sample_08679.html,2314,523,0.421321,5.222944,3,17,0.071998,5.370348,1,196,12716,7,553,0,0


Excluding the `TARGET` / `label` column in train data and the `file_name` which represents the document identifier, you can see that there are 13 types of static HTML features.

Let's review the column types, non-null counts, and summary statistics:

In [8]:
# Check column information and missing values
print("--- Train Data Information ---")
train.info()

print("\n--- Missing Values Summary ---")
missing_count = train.isnull().sum().sum()
print(f"Total missing values in train dataset: {missing_count} (100% complete)")

--- Train Data Information ---
<class 'pandas.DataFrame'>
RangeIndex: 11998 entries, 0 to 11997
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   file_name                11998 non-null  str    
 1   url_punct_char_count     11998 non-null  int64  
 2   tag_count                11998 non-null  int64  
 3   whitespace_ratio         11998 non-null  float64
 4   entropy                  11998 non-null  float64
 5   form_count               11998 non-null  int64  
 6   embedded_js_count        11998 non-null  int64  
 7   html_whitespace_ratio    11998 non-null  float64
 8   script_entropy           11998 non-null  float64
 9   min_link_length          11998 non-null  int64  
 10  external_link_count      11998 non-null  int64  
 11  total_script_characters  11998 non-null  int64  
 12  internal_link_count      11998 non-null  int64  
 13  url_digit_count          11998 non-null  int64  
 14  la

In [9]:
# Summary statistics of the 13 numeric features
feature_list = [
    "url_punct_char_count",
    "tag_count",
    "whitespace_ratio",
    "entropy",
    "form_count",
    "embedded_js_count",
    "html_whitespace_ratio",
    "script_entropy",
    "min_link_length",
    "external_link_count",
    "total_script_characters",
    "internal_link_count",
    "url_digit_count"
]

train[feature_list].describe().T

,count,mean,std,min,25%,50%,75%,max
url_punct_char_count,11998.0,841.381147,1861.384716,0.000000,47.000000,265.000000,1028.750000,8.360500e+04
tag_count,11998.0,485.531172,978.012871,3.000000,57.000000,186.000000,601.750000,4.359100e+04
whitespace_ratio,11998.0,0.372521,0.165130,0.000000,0.285080,0.397851,0.488199,8.543541e-01
entropy,11998.0,5.162909,0.388353,0.914438,4.989247,5.175101,5.352082,7.874702e+00
form_count,11998.0,1.323721,2.949107,0.000000,0.000000,1.000000,1.000000,1.890000e+02
embedded_js_count,11998.0,7.271962,12.560264,0.000000,1.000000,3.000000,9.000000,3.330000e+02
html_whitespace_ratio,11998.0,0.119085,0.076704,0.000010,0.070630,0.108801,0.150384,8.300951e-01
script_entropy,11998.0,4.103867,2.059690,0.000000,4.316360,5.031891,5.367841,6.274234e+00
min_link_length,11998.0,21.327721,777.871013,0.000000,1.000000,1.000000,11.000000,7.769800e+04
external_link_count,11998.0,54.984747,114.434499,0.000000,1.000000,16.000000,67.000000,5.048000e+03


The 13 extracted features capture structural, lexical, script, and URL characteristics designed to separate phishing artifacts from benign web content without requiring code execution:

| # | Feature Name | Description & Threat Relevance |
| :--- | :--- | :--- |
| 1 | `url_punct_char_count` | Punctuation characters in URLs. Phishing links often employ nested redirects, base64 query tokens, or deceptive subdomains with heavy punctuation. |
| 2 | `tag_count` | Total count of HTML tags in the DOM. Legitimate portals have rich DOM structures (median ~514 tags), whereas phishing templates tend to be minimal landing pages (median ~91 tags). |
| 3 | `whitespace_ratio` | Ratio of whitespace in extracted text. Phishers often introduce abnormal whitespace padding to bypass simple keyword filters. |
| 4 | `entropy` | Shannon entropy of the overall raw HTML document. Obfuscated, packed, or encoded HTML shows distinctive entropy signatures. |
| 5 | `form_count` | Total `<form>` elements. Phishing attacks almost universally revolve around harvesting credentials (usernames, passwords, MFA codes). |
| 6 | `embedded_js_count` | Number of inline `<script>` blocks (scripts without an external `src`). Inline scripts are commonly used to execute browser fingerprinting without external domain lookups. |
| 7 | `html_whitespace_ratio` | Ratio of whitespace across the full HTML source code, reflecting code minification or packing patterns. |
| 8 | `script_entropy` | Shannon entropy calculated specifically over JavaScript code blocks. Packed or encrypted JS payloads produce noticeably higher entropy. |
| 9 | `min_link_length` | Character length of the shortest hyperlink in the page. Single-character links (`#`, `/`) are common in decoy templates. |
| 10 | `external_link_count` | Number of hyperlinks referencing external domains. Legitimate sites have diverse external navigation, while phishing kits restrict external links. |
| 11 | `total_script_characters`| Cumulative character count of all JavaScript blocks, distinguishing rich web apps from compact phishing capture scripts. |
| 12 | `internal_link_count` | Number of internal or same-domain hyperlinks. Legitimate sites feature extensive site navigation, while phishing sites rarely implement internal links. |
| 13 | `url_digit_count` | Total numeric digits across all hyperlinks, frequently elevated when IP addresses or randomized session tokens are used in domain names. |

### 2.2 Selecting Features

It is often difficult to perform data analysis and preprocessing on all features from the beginning. Instead, an easier way to get started is to start with a small number of features and then add features one by one.

This notebook will focus on the top HTML features. For the remaining features, please refer to the methods introduced in this notebook, etc., and perform the analysis on your own.

In [10]:
# Focus on the top HTML features
use_features = [
    "url_punct_char_count",
    "tag_count",
    "whitespace_ratio",
    "entropy",
    "form_count",
    "embedded_js_count",
    "html_whitespace_ratio",
    "script_entropy",
    "min_link_length",
    "external_link_count",
    "total_script_characters",
    "internal_link_count",
    "url_digit_count",
    # add new features to use here
]
target = train["TARGET"].values

train = train[use_features]
train["TARGET"] = target
test = test[use_features]

Let's check the data once again.

In [11]:
# Check train data
print(f"train shape: {train.shape}")
train.head(3)

train shape: (11998, 14)


,url_punct_char_count,tag_count,whitespace_ratio,entropy,form_count,embedded_js_count,html_whitespace_ratio,script_entropy,min_link_length,external_link_count,total_script_characters,internal_link_count,url_digit_count,TARGET
0,2095,756,0.416768,5.465578,3,73,0.066372,5.539064,8,221,89458,21,1011,0
1,107,106,0.345622,5.382703,1,1,0.034719,5.368758,5,15,7128,1,126,1
2,1681,1182,0.359943,5.399269,5,4,0.091763,5.496770,9,172,13840,32,2065,0


In [12]:
# Check test data
print(f"test shape: {test.shape}")
test.head(3)

test shape: (7999, 13)


,url_punct_char_count,tag_count,whitespace_ratio,entropy,form_count,embedded_js_count,html_whitespace_ratio,script_entropy,min_link_length,external_link_count,total_script_characters,internal_link_count,url_digit_count
0,0,5,0.000000,5.375038,0,1,0.064619,5.309058,0,0,1713,0,0
1,525,383,0.360669,5.282739,5,23,0.103680,5.381425,19,61,28705,7,23
2,2314,523,0.421321,5.222944,3,17,0.071998,5.370348,1,196,12716,7,553
